# 00 先进文生图（Kaggle 2026 加固）
**严禁乱升 numpy**。模型：FLUX → SD3.5 → SDXL-Turbo → SDXL → 程序化回退

In [ ]:
import os, sys, gc, math, subprocess
from pathlib import Path
import torch
print(sys.version, torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available()
print('GPU', torch.cuda.get_device_name(0))
GEN=Path('/kaggle/working/gen_images')
for s in ['single_object','multiview/set_001','exports']:
    (GEN/s if '/' not in s else GEN.joinpath(*s.split('/'))).mkdir(parents=True, exist_ok=True)

def pip_install(pkgs, no_deps=False):
    cmd=[sys.executable,'-m','pip','install','-q']
    if no_deps: cmd.append('--no-deps')
    cmd += pkgs
    print('>>', cmd)
    r=subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print((r.stderr or r.stdout or '')[-1500:])
    return r.returncode==0

# 只装扩散栈，不碰 numpy
pip_install(['diffusers','transformers','accelerate','safetensors','sentencepiece','huggingface_hub','Pillow'])
# peft 可选
pip_install(['peft'])

import numpy as np
print('numpy', np.__version__)
try:
    import diffusers, transformers
    print('diffusers', diffusers.__version__, 'transformers', transformers.__version__)
except Exception as e:
    print('import after install failed', e)

In [ ]:
import os, gc, torch
from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np

HF_TOKEN=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    if not HF_TOKEN:
        HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe=None; MODEL_ID=None; LOG=[]

def offload(p):
    try: p.enable_model_cpu_offload()
    except Exception:
        try: p.to('cuda')
        except Exception: pass
    return p

def try_load(tag, loader):
    global pipe, MODEL_ID
    print('try', tag)
    try:
        gc.collect(); torch.cuda.empty_cache()
        p, mid = loader()
        pipe, MODEL_ID = offload(p), mid
        LOG.append('OK '+tag); print('USING', mid); return True
    except Exception as e:
        LOG.append('FAIL '+tag+': '+str(e)[:300]); print('FAIL', tag, str(e)[:400]); pipe=None; return False

def load_flux():
    from diffusers import FluxPipeline
    mid='black-forest-labs/FLUX.1-schnell'
    return FluxPipeline.from_pretrained(mid, torch_dtype=DTYPE, token=HF_TOKEN), mid

def load_sd35():
    from diffusers import StableDiffusion3Pipeline
    mid='stabilityai/stable-diffusion-3.5-medium'
    return StableDiffusion3Pipeline.from_pretrained(mid, torch_dtype=DTYPE, token=HF_TOKEN), mid

def load_turbo():
    from diffusers import AutoPipelineForText2Image
    mid='stabilityai/sdxl-turbo'
    p=AutoPipelineForText2Image.from_pretrained(mid, torch_dtype=DTYPE)
    p.to('cuda'); return p, mid

def load_sdxl():
    from diffusers import DiffusionPipeline
    mid='stabilityai/stable-diffusion-xl-base-1.0'
    p=DiffusionPipeline.from_pretrained(mid, torch_dtype=DTYPE, use_safetensors=True)
    return p, mid

for tag,fn in [('flux',load_flux),('sd35',load_sd35),('turbo',load_turbo),('sdxl',load_sdxl)]:
    if try_load(tag, fn):
        break
print('LOAD_LOG', LOG)

In [ ]:
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageFilter
import numpy as np
import torch, shutil

GEN=Path('/kaggle/working/gen_images')
SUBJECT='a ceramic coffee mug with a small chip on the rim'
STYLE='studio product photo, centered object, full object visible, plain pure white background, soft even lighting, sharp focus, photorealistic, no text'
NEG='blurry, lowres, cropped, deformed, multiple objects, busy background, watermark, text'

def procedural(view='front', seed=0, size=768):
    rng=np.random.default_rng(seed)
    img=Image.new('RGB',(size,size),(250,250,250))
    d=ImageDraw.Draw(img)
    # mug body
    cx,cy=size//2, size//2+20
    if view=='top':
        d.ellipse((cx-120,cy-90,cx+120,cy+90), fill=(200,120,80), outline=(120,70,40), width=4)
        d.ellipse((cx-70,cy-50,cx+70,cy+50), fill=(230,200,170))
    elif view in ('side_left','side_right','back'):
        d.rounded_rectangle((cx-90,cy-140,cx+90,cy+130), radius=40, fill=(200,120,80), outline=(120,70,40), width=4)
        if view!='back':
            hx = cx+90 if view=='side_left' else cx-90
            d.arc((hx-40,cy-40,hx+40,cy+60), 270, 90, fill=(120,70,40), width=10)
    else:
        d.rounded_rectangle((cx-100,cy-140,cx+100,cy+130), radius=45, fill=(200,120,80), outline=(120,70,40), width=4)
        d.arc((cx+70,cy-30,cx+150,cy+70), 270, 90, fill=(120,70,40), width=12)
        # chip
        d.ellipse((cx-30,cy-145,cx-5,cy-125), fill=(250,250,250))
    # light noise for non-flat texture
    arr=np.array(img).astype(np.int16)
    arr += rng.integers(-3,4, arr.shape, dtype=np.int16)
    arr=np.clip(arr,0,255).astype(np.uint8)
    return Image.fromarray(arr)

def generate_one(prompt, neg=None, seed=42, size=768):
    if pipe is None:
        # parse view hint
        view='front'
        for v in ['front','side_left','side_right','back','three_quarter','top']:
            if v.replace('_',' ') in prompt or v in prompt: view=v; break
        return procedural(view=view, seed=seed, size=size)
    g=torch.Generator(device='cuda').manual_seed(seed)
    kwargs=dict(prompt=prompt, generator=g)
    mid=(MODEL_ID or '').lower()
    if 'flux' in mid:
        kwargs.update(num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256, height=size, width=size)
    elif 'turbo' in mid:
        kwargs.update(num_inference_steps=4, guidance_scale=0.0, height=size, width=size)
    elif 'stable-diffusion-3' in mid:
        kwargs.update(num_inference_steps=28, guidance_scale=4.5, height=size, width=size)
        if neg: kwargs['negative_prompt']=neg
    else:
        kwargs.update(num_inference_steps=30, guidance_scale=7.0, height=size, width=size)
        if neg: kwargs['negative_prompt']=neg
    with torch.inference_mode():
        return pipe(**kwargs).images[0]

prompt=f'{SUBJECT}, {STYLE}'
print('MODEL', MODEL_ID)
img=generate_one(prompt, NEG, 42, 768)
hero=GEN/'single_object'/'hero_000.png'; img.save(hero); print('saved', hero)
try: display(img.resize((320,320)))
except Exception: pass

VIEWS=[('front','front view'),('side_left','left side view'),('side_right','right side view'),
       ('back','back view'),('three_quarter','three-quarter view 45 degree'),('top','top-down view')]
mv=GEN/'multiview'/'set_001'; paths=[]
for i,(name,hint) in enumerate(VIEWS):
    p=f'{SUBJECT}, {hint}, {STYLE}, same object identity'
    print('gen', name)
    im=generate_one(p, NEG, 1000+i, 768)
    fp=mv/f'{i:02d}_{name}.png'; im.save(fp); paths.append(fp)
    try: display(im.resize((160,160)))
    except Exception: pass
(mv/'manifest.txt').write_text('\n'.join(map(str,paths)), encoding='utf-8')
zb=GEN/'exports'/f'gen_images_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
shutil.make_archive(str(zb),'zip', root_dir=GEN)
print('00 DONE model=', MODEL_ID, 'zip=', str(zb)+'.zip', 'views', len(paths))
assert hero.exists() and len(paths)==6